# പാഠം 11 - മോഡൽ കോൺടെക്സ്റ്റ് പ്രോട്ടോക്കോൾ (MCP)

**മോഡൽ കോൺടെക്സ്റ്റ് പ്രോട്ടോക്കോൾ (MCP)** ഒരു തുറന്ന സ്റ്റാൻഡേർഡാണ്, ഇത് എജന്റുകൾക്ക് റൺടൈമിൽ ഡൈനാമിക്കായി ഉപകരണങ്ങൾ, വിഭവങ്ങൾ, ഡാറ്റാ സ്രോതസ്സുകൾ കണ്ടെത്താനും ഉപയോഗിക്കാനുമുള്ള സൗകര്യം നൽകുന്നു. ഒരു എജന്റിന്റെ ഉള്ളിലേക്ക് ഉപകരണങ്ങൾ ഹാർഡ്‌കോഡ് ചെയ്യുന്നതിന് പകരം, MCP ആവശ്യാനുസരിച്ച് കഴിവുകൾ പുറത്തുവിടുന്ന ബാഹ്യ സർവറുകളുമായി എജന്റുകൾക്ക് ബന്ധപ്പെടാൻ അനുവദിക്കുന്നു.

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിക്കും:
- MCP എന്താണെന്നും ഏജന്റ് സിസ്റ്റങ്ങൾക്കായുള്ള അതിന്റെ പ്രാധാന്യവും
- MCP ക്ലയന്റ്-സർവർ ആർക്കിടെക്ചർ എങ്ങനെ പ്രവർത്തിക്കുന്നു
- MCP-ശൈലിയിലെ ഉപകരണങ്ങളുടെ കണ്ടെത്തൽ ഉപയോഗിക്കുന്ന എജന്റുകൾ എങ്ങനെ നിർമ്മിക്കാം


## ക്രമീകരണം

**ആവശ്യകതകൾ:**
- ഡിപ്ലോയുചെയ്ത മോഡൽ ഉള്ള Azure AI Foundry പ്രോജക്റ്റ്
- പ്രാമാണീകരണത്തിന് `az login` ഓടിക്കുക


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## മോഡൽ കോൺടെക്സ്റ്റ് പ്രോട്ടോക്കോൾ (MCP) എന്താണ്?

MCP എഐ ഏജന്റുകൾക്ക് ബാഹ്യ ഉപകരണങ്ങളും ഡാറ്റാ സ്രോതസ്സുകളും കണ്ടെത്തി അവയുമായി ഇടപഴകാനുള്ള ഒരു സ്റ്റാൻഡേർഡ് രീതിയാണ് നിർവചിക്കുന്നത്:

- **MCP സെർവർ**: ഒരു സ്റ്റാൻഡേർഡ് പ്രോട്ടോക്കോളിലൂടെ ഉപകരണങ്ങളും വിഭവങ്ങളും പ്രോംപ്റ്റുകളും ലഭ്യമാക്കുന്നു
- **MCP ക്ലയന്റ്**: ഇത് സർവറുകളുമായി കണക്ട് ചെയ്ത് ലഭ്യമായ കഴിവുകൾ കണ്ടെത്തുന്ന ഏജന്റ് റൺടൈമാണ്
- **ഡൈനാമിക് കണ്ടെത്തൽ**: ഏജന്റുകൾക്ക് ഹാർഡ്കോഡഡ് ടൂളുകൾ ആവശ്യമില്ല — റൺടൈമിൽ എന്താണ് ലഭ്യമെന്നത് അവ കണ്ടെത്തുന്നു

ഇത് പുതിയ ശേഷികൾ ഏജന്റ് കോഡ് മാറ്റാതെ ചേർക്കാൻ കഴിയുന്നതിനാൽ വിപുലീകരിക്കാവുന്ന ഏജന്റ് സിസ്റ്റങ്ങൾ നിർമ്മിക്കാൻ ശക്തമായ മാർഗമാണ്.


## MCP എങ്ങനെ പ്രവർത്തിക്കുന്നു

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ഏജന്റ് (MCP ക്ലയന്റ്) ഒരു MCP സെർവറുമായി ബന്ധപ്പെടുന്നു
2. സെർവർ ലഭ്യമായ ടൂളുകളുടെയും അവയുടെ സ്കീമകളുടെയും പട്ടികയുമായി പ്രതികരിക്കുന്നു
3. ഏജന്റ് പിന്നീട് തന്റെ നിഗമനപ്രക്രിയയിൽ കണ്ടെത്തിയ ഏതെങ്കിലും ടൂൾ വിളിക്കാം
4. ഫലങ്ങൾ അതേ പ്രോട്ടോക്കോളിലൂടെ മടങ്ങി വരുന്നു


## MCP ടൂൾ കണ്ടെത്തലിന്റെ സിമുലേഷൻ

യാഥാർത്ഥ്യമായ MCP സെർവർ ഒരു പ്രവർത്തിക്കുന്ന സെർവർ പ്രക്രിയയെ ആവശ്യമാക്കുന്നതുകൊണ്ട്, MCP-യോട് ബന്ധമുള്ള വാസസൗകര്യ സേവനം നൽകുന്നതിനെ അനുകരിക്കുന്ന `@tool` ഫംഗ്ഷനുകൾ ഉപയോഗിച്ച് നമ്മൾ ഈ പാറ്റേൺ കാണിക്കുന്നു.

ഉത്പാദന സാഹചര്യങ്ങളിൽ, ഈ ടൂളുകൾ ലോക്കലായി നിർവചിക്കപ്പെടുന്നതിന് പകരം MCP സെർവറിൽ നിന്ന് ഡൈനാമിക്കായി കണ്ടെത്തപ്പെടും.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ശൈലി ഉപകരണങ്ങളോടുകൂടി ഒരു ഏജന്റ് നിർമ്മിക്കൽ


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP ഉത്പാദനത്തിൽ

ഉത്പാദന പരിസരത്തിൽ, MCP ശക്തമായ മാതൃകകൾ സാധ്യമാക്കുന്നു:

- **ഡൈനാമിക് ടൂൾ കണ്ടെത്തൽ**: ഏജന്റുകൾ MCP സെർവറുകളുമായി ബന്ധിപ്പിച്ച് റൺടൈമിൽ ടൂളുകൾ കണ്ടെത്തുന്നു
- **വിയുക്ത ഘടന**: ടൂൾ പ്രൊവൈഡർമാർ ഏജന്റിൽ നിന്ന് സ്വതന്ത്രമായി പുതുക്കാൻ കഴിയും
- **സംഘടനകൾ തമ്മിലുള്ള പങ്കുവെയ്ക്കൽ**: ടീമുകൾ MCP സെർവറുകൾ വഴി ഏജന്റുകൾക്ക് ഉപയോഗിക്കാവുന്ന കഴിവുകൾ പങ്കുവെക്കാം
- **Microsoft Agent Framework പിന്തുണ**: MAF-ൽ `mcp` ഇന്റഗ്രേഷന്റെ വഴി ഉൾപ്പെടുത്തിയ MCP ക്ലയന്റ് പിന്തുണ ഉണ്ട്

MAF-നൊപ്പം യഥാർത്ഥ MCP സെർവർ ഉപയോഗിക്കാൻ, നിങ്ങൾ `hosted_mcp_tool()` അല്ലെങ്കിൽ MCP ക്ലയന്റ് ഇന്റഗ്രേഷൻ വഴി ബന്ധിപ്പിക്കണം.

**കൂടുതൽ അറിയുക:**
- [MCP സ്പെസിഫിക്കേഷൻ](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP പിന്തുണ](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:
- **MCP** ഏജന്റുകളും ടൂൾ പ്രൊവൈഡർമാരും തമ്മിലുള്ള ഡൈനാമിക് ടൂൾ കണ്ടെത്തലിനുള്ള ഒരു തുറന്ന സ്റ്റാൻഡേർഡാണ്
- **ക്ലയന്റ്-സെർവർ ആർക്കിടെക്ചർ** ഏജന്റുകൾക്ക് റൺടൈമിൽ കഴിവുകൾ കണ്ടെത്താൻ അനുവദിക്കുന്നു
- MCP **വിസ്താരയോഗ്യമായ, വേർതിരിച്ച എജന്റ് സിസ്റ്റങ്ങൾ** സജ്ജമാക്കുന്നു, ഇവയിൽ ടൂളുകൾ കോഡ് മാറ്റങ്ങളില്ലാതെ ചേർക്കാവുന്നതാണ്
- Microsoft Agent Framework ഉത്പാദന ഉപയോഗത്തിനായി **നിർമിത MCP പിന്തുണ** നൽകുന്നു


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ഡിസ്ക്ലെയിമർ:
ഈ ഡോക്യുമെന്റ് AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. ഞങ്ങൾ ശുദ്ധതയിലും ശരിതീരത്തിലും ശ്രമിച്ചുതന്നെ മുന്നോട്ട് പോകുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് വിവർത്തനങ്ങളിൽ പിശകുകൾ അല്ലെങ്കിൽ അശുദ്ധതകൾ ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. യഥാർത്ഥ രേഖയും അതിന്റെ മാതൃഭാഷയിലുളള പതിപ്പും പ്രാമാണിക ഉറവിടമായിരിക്കുന്നുവെന്ന് കരുതുക. ഗൗരവമുള്ള/പ്രധാന വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനത്തിന്റെ ഉപയോഗം മൂലം ഉണ്ടാകുന്ന ഏതെങ്കിലും തരത്തിലുള്ള തെറ്റിദ്ധാരണകൾക്കുമരു തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കുമാണ് ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
